In [1]:
# mlpのparamsを計算する関数
def mlp_params(input_dim,output_dim):
    # hidden size 1024, layers 2
    hidden_size = 1024
    layer1 = (input_dim * hidden_size) + hidden_size
    layer2 = (hidden_size * hidden_size) + hidden_size
    layer3 = (hidden_size * output_dim) + output_dim
    params = layer1 + layer2 + layer3
    return params

# rnnのparamsを計算する関数
def rnn_params(input_dim,output_dim):
    # hidden size 400, layers 2, LSTM Cell
    hidden_size = 400
    layer1 = 4 * ((input_dim + hidden_size) * hidden_size + hidden_size)
    layer2 = 4 * ((hidden_size + hidden_size) * hidden_size + hidden_size)
    layer3 = (hidden_size * output_dim) + output_dim
    params = layer1 + layer2 + layer3
    return params

# ncpのparamsを計算する関数
def ncp_params(input_dim, output_dim, units):
    layer1 = (input_dim * units) + units
    params_1 = units * 3 #gleak,vleak,cm
    params_2 = units * units * 5 # sigma, mu, w, erev, sparsity_mask
    params_3 = input_dim * units * 5 # sensory_sigma, sensory_mu, sensory_w, sensory_erev, sensory_sparsity_mask
    layer2 = (units * output_dim) + output_dim
    params = layer1 + params_1 + params_2 + params_3 + layer2
    return params

import re
def calculate_model_size(df, task):
    df = df.copy()
    if task == "lift": #[10,3,4,2]
        input_dim = 19
        output_dim = 7
    elif task == "can": #[14,3,4,2]
        input_dim = 23
        output_dim = 7
    elif task == "square": #[14,3,4,2]
        input_dim = 23
        output_dim = 7
    else:
        raise ValueError("Unknown task")
    for model in df["model"].unique():
        idx = df["model"] == model
        if model == "bc-pure":
            df.loc[idx, "params"] = mlp_params(input_dim, output_dim)
        elif model == "bc-rnn-pure":
            df.loc[idx, "params"] = rnn_params(input_dim, output_dim)
        elif model.startswith("ncp"):
            u = int(re.search(r"_u(\d+)", model).group(1))
            df.loc[idx, "params"] = ncp_params(input_dim, output_dim, u)
        else:
            raise ValueError("Unknown model")
    return df

In [33]:
import re
import csv
import numpy as np
import pandas as pd
from pathlib import Path

# -------------------------------------------------
# CSV を列ズレ耐性付きで読む
# -------------------------------------------------
def robust_read_csv(path: str) -> pd.DataFrame:
    with open(path, "r") as f:
        lines = [ln.rstrip("\n") for ln in f if ln.strip()]

    if not lines:
        return pd.DataFrame()

    header = next(csv.reader([lines[0]]))
    ncols = len(header)

    rows = []
    for ln in lines[1:]:
        fields = next(csv.reader([ln]))

        if len(fields) > ncols:
            extra = len(fields) - ncols
            name = ",".join(fields[: extra + 1])
            rest = fields[extra + 1 :]
            fields = [name] + rest
        elif len(fields) < ncols:
            fields = fields + [None] * (ncols - len(fields))

        rows.append(fields)

    return pd.DataFrame(rows, columns=header)


# -------------------------------------------------
# name 解析系
# -------------------------------------------------
def extract_success_from_name(name: str) -> float:
    m = re.search(r"(?:^|[_-])success[_-]([0-9]*\.?[0-9]+)", str(name))
    return float(m.group(1)) if m else np.nan


def extract_seed_last(name: str) -> float:
    seeds = re.findall(r"(?:^|[_-])seed(\d+)", str(name))
    return float(seeds[-1]) if seeds else np.nan


# -------------------------------------------------
# 単一 CSV → top-k 平均
# -------------------------------------------------
def get_topk_avg_score(
    df: pd.DataFrame,
    seed_min: int = 1,
    seed_max: int = 20,
    model_name: str | None = None,
    top_k: int = 10,
    rank_by: str = "success_rate",
) -> pd.DataFrame:

    df = df.copy()

    if "name" not in df.columns:
        raise KeyError("'name' column not found")

    # 数値化
    for col in ["success_rate", "return"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # success_rate を name から復元
    if "success_rate" in df.columns:
        sr_from_name = df["name"].map(extract_success_from_name)
        mask = sr_from_name.notna()
        df.loc[mask, "success_rate"] = sr_from_name[mask]

        df = df[
            df["success_rate"].between(0.0, 1.0, inclusive="both")
            | df["success_rate"].isna()
        ]

    # seed 抽出
    df["seed"] = df["name"].map(extract_seed_last)
    df = df[df["seed"].between(seed_min, seed_max, inclusive="both")]

    # rank_by 欠損除去
    df = df[df[rank_by].notna()]

    # model 名
    df["model"] = model_name if model_name else (
        df["name"].astype(str).str.replace(r"(?:_seed|seed)\d+", "", regex=True)
    )

    # seed ごとに最大 success_rate を取る
    df_seed = (
        df.groupby(["model", "seed"], as_index=False)
        .agg(
            success_rate_max=("success_rate", "max"),
            return_mean=("return", "mean"),
        )
    )

    # top-k seed
    df_seed = df_seed.sort_values(
        ["model", "success_rate_max"], ascending=[True, False]
    )
    df_top = df_seed.groupby("model", as_index=False).head(top_k)

    # model 平均（← ここで「平均成功率」が明示的に出る）
    df_avg = (
        df_top.groupby("model")
        .agg(
            success_rate_avg=("success_rate_max", "mean"),
            return_avg=("return_mean", "mean"),
            n_seeds=("seed", "nunique"),
            selected_seeds=("seed", lambda s: sorted(map(int, s.tolist()))),
        )
        .reset_index()
    )

    return df_avg

# -------------------------------------------------
# 複数 CSV
# -------------------------------------------------
def aggregate_multiple_csvs(
    csv_files: list[str],
    seed_min: int = 1,
    seed_max: int = 20,
    top_k: int = 10,
    rank_by: str = "success_rate",
) -> pd.DataFrame:

    results = []

    for path in csv_files:
        print(f"\n##### {Path(path).name} (debug head) #####")
        df = robust_read_csv(path)
        if df.empty:
            continue

        # --- デバッグ表示（必要最小限） ---
        debug_cols = [c for c in ["name", "return", "horizon"] if c in df.columns]
        print(df[debug_cols].head(2))
        print(f"rows: {len(df)}")

        df_avg = get_topk_avg_score(
            df,
            seed_min=seed_min,
            seed_max=seed_max,
            model_name=Path(path).stem,
            top_k=top_k,
            rank_by=rank_by,
        )

        df_avg["source_csv"] = Path(path).name
        results.append(df_avg)

    if not results:
        return pd.DataFrame()

    return pd.concat(results, ignore_index=True)


# -------------------------------------------------
# 使い方
# -------------------------------------------------
csv_files = [
    "/work/robomimic/csv/result/ncp_best/lift_u64.csv",
    "/work/robomimic/csv/result/ncp_best/lift_u128.csv",
    "/work/robomimic/csv/result/ncp_best/lift_u256.csv",
]

df_all = aggregate_multiple_csvs(
    csv_files,
    top_k=10,
    rank_by="success_rate",  # ← "return" に切り替えてもOK
)

print("\n=== ALL RESULTS ===")
print(df_all)



##### lift_u64.csv (debug head) #####
    name return horizon
0  seed1   0.99   51.92
1  seed2   0.98   51.06
rows: 20

##### lift_u128.csv (debug head) #####
                  name return horizon
0  ncp_u128_best_seed1   0.98   52.33
1  ncp_u128_best_seed2   0.96   66.58
rows: 20

##### lift_u256.csv (debug head) #####
                  name return horizon
0  ncp_u256_best_seed2    1.0   46.02
1  ncp_u256_best_seed3    1.0   45.99
rows: 20

=== ALL RESULTS ===
       model  success_rate_avg  return_avg  n_seeds  \
0   lift_u64             0.988      43.184       10   
1  lift_u128             0.995      46.849       10   
2  lift_u256             0.992      36.981       10   

                         selected_seeds     source_csv  
0  [1, 2, 8, 9, 12, 14, 16, 17, 18, 20]   lift_u64.csv  
1  [4, 6, 7, 8, 10, 12, 14, 15, 16, 17]  lift_u128.csv  
2    [2, 3, 5, 6, 8, 9, 10, 15, 18, 19]  lift_u256.csv  
